# Nobody hands you a diagram

The previous notebook (`nbs/diagnose/06`) took a graph someone drew, tested the
claims it made, and repaired the one the data denied. That repaired graph
survived — which is worth a lot, and is still one graph.

This one asks the other question. What would the data have said on its own?
How much of that answer is real rather than an artifact of the particular rows
you happened to collect? And what happens when the thing driving everything was
never measured at all?

In [ ]:
import numpy as np
import plotly.graph_objects as go

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import AQUA, AXIS, BLUE, CRITICAL, GOOD, GRID, INK, MUTED, ORANGE, SURFACE
from _style import SUBTLE as SECOND
from _style import caption, compare

NODE_FILL, NODE_RING = "#ffffff", AXIS
RADIUS = 0.15


def _rim(x0, y0, x1, y1, back=RADIUS):
    dx, dy = x1 - x0, y1 - y0
    length = np.hypot(dx, dy) or 1.0
    ux, uy = dx / length, dy / length
    return x0 + ux * back, y0 + uy * back, x1 - ux * back, y1 - uy * back


def _nodes(fig, positions):
    names = list(positions)
    fig.add_trace(go.Scatter(
        x=[positions[n][0] for n in names], y=[positions[n][1] for n in names],
        mode="markers+text", text=names, textposition="middle center",
        textfont=dict(size=12, color=INK),
        marker=dict(size=52, color=NODE_FILL, line=dict(color=NODE_RING, width=1.5)),
        hoverinfo="skip", showlegend=False,
    ))


def _frame(fig, positions, title, subtitle, height):
    """Fit the axes to the nodes, so the drawing fills the canvas it is given."""
    xs = [p[0] for p in positions.values()]
    ys = [p[1] for p in positions.values()]
    pad_x = max(0.35, (max(xs) - min(xs)) * 0.18)
    pad_y = max(0.35, (max(ys) - min(ys)) * 0.25)
    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color=INK), x=0.01,
                   subtitle=dict(text=subtitle, font=dict(size=12, color=SECOND))),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, height=height,
        margin=dict(l=10, r=10, t=64, b=10),
        xaxis=dict(visible=False, range=[min(xs) - pad_x, max(xs) + pad_x]),
        yaxis=dict(visible=False, range=[min(ys) - pad_y, max(ys) + pad_y]),
    )


def pag_figure(positions, pag, title, subtitle="", height=360):
    """Draw a PAG with its three end-marks: arrowhead, tail, and open circle."""
    fig = go.Figure()
    circles_x, circles_y = [], []
    for edge in pag.edges:
        x0, y0, x1, y1 = _rim(*positions[edge.a], *positions[edge.b])
        colour = CRITICAL if edge.is_bidirected else (BLUE if edge.is_directed else MUTED)
        fig.add_shape(type="line", x0=x0, y0=y0, x1=x1, y1=y1,
                      line=dict(color=colour, width=2))
        for mark, (px, py), (qx, qy) in (
            (edge.mark_a, (x0, y0), (x1, y1)),
            (edge.mark_b, (x1, y1), (x0, y0)),
        ):
            if mark == "arrow":       # a short arrow landing on the node
                dx, dy = px - qx, py - qy
                length = np.hypot(dx, dy) or 1.0
                fig.add_annotation(
                    x=px, y=py, ax=px - dx / length * 0.16, ay=py - dy / length * 0.16,
                    xref="x", yref="y", axref="x", ayref="y",
                    showarrow=True, arrowhead=2, arrowsize=1.3, arrowwidth=2,
                    arrowcolor=colour, standoff=0,
                )
            elif mark == "circle":    # an open circle: the data did not decide
                circles_x.append(px)
                circles_y.append(py)
    if circles_x:
        fig.add_trace(go.Scatter(
            x=circles_x, y=circles_y, mode="markers", showlegend=False, hoverinfo="skip",
            marker=dict(size=11, color=SURFACE, line=dict(color=MUTED, width=2)),
        ))
    _nodes(fig, positions)
    _frame(fig, positions, title, subtitle, height)
    return fig

In [ ]:
from axiom.discover import Dataset

rng = np.random.default_rng(11)
n = 2000
light = rng.normal(size=n)
water = rng.normal(size=n)
heat = 1.3 * light + rng.normal(size=n)
growth = 0.9 * light + 0.8 * water + 1.1 * heat + rng.normal(size=n)
yield_ = 1.4 * growth + rng.normal(size=n)

trial = Dataset.observational(
    np.column_stack([light, heat, water, growth, yield_]),
    ["light", "heat", "water", "growth", "yield"],
)
POSITIONS = {
    "light": (0.0, 1.05), "heat": (1.0, 1.3), "water": (0.0, 0.0),
    "growth": (1.15, 0.55), "yield": (2.1, 0.55),
}
print(trial.n_rows, "rows ×", len(trial.names), "variables")

## What one run says

`ges` searches equivalence classes, so what it returns is a *class*: an edge is
directed only where every graph in the class agrees. Undirected edges are not
a failure of the search — they are the honest report that observational data
cannot orient them.

In [ ]:
from axiom.discover import GaussianBIC, ges

found = ges(GaussianBIC(trial))
print(found.essential.to_text())
print(found.summary())

## How much of that survives resampling

One run is one sample. `edge_stability` resamples the rows and re-runs the
search, counting how often each edge comes back — and, when it does, which way
it pointed.

The three-way split matters more than the total. An edge can be shaky in two
completely different ways, and they call for opposite responses.

In [ ]:
from axiom.discover import EdgeSupport, StabilityReport, edge_stability

stability = edge_stability(trial, n_bootstrap=60, seed=7)
assert isinstance(stability, StabilityReport)
print(stability.summary(), "\n")
rows = []
for support in stability.edges:
    assert isinstance(support, EdgeSupport)
    rows.append([f"{support.a} – {support.b}", support.describe()])
table(rows, headers=("edge", "support across resamples"))

In [ ]:
shown = [e for e in stability.edges if e.adjacent >= 0.05]
labels = [f"{e.a}–{e.b}" for e in shown]
# The row label reads "a-b", so name the directions by it rather than by
# "one way"/"the other", which a reader cannot resolve.
series = [
    ("left → right", [e.forward for e in shown], BLUE),
    ("right → left", [e.backward for e in shown], ORANGE),
    ("present, not oriented", [e.undirected for e in shown], AQUA),
]

fig = go.Figure()
for name, values, colour in series:
    fig.add_trace(go.Bar(
        y=labels, x=values, name=name, orientation="h", marker_color=colour,
        marker_line=dict(color=SURFACE, width=2),      # a 2px surface gap between fills
        text=[f"{v:.0%}" if v >= 0.12 else "" for v in values],
        textposition="inside", insidetextfont=dict(color="#ffffff", size=11),
        hovertemplate="%{y}<br>" + name + ": %{x:.0%}<extra></extra>",
    ))
fig.update_layout(
    barmode="stack",
    title=dict(text="How often each edge came back, over 60 resamples",
               font=dict(size=15, color=INK), x=0.01,
               subtitle=dict(text="Full bars are edges the data is sure of. "
                                  "Aqua means sure it is there, unsure which way it runs.",
                             font=dict(size=12, color=SECOND))),
    xaxis=dict(title="share of resamples", tickformat=".0%", range=[0, 1.02],
               gridcolor=GRID, color=SECOND),
    yaxis=dict(autorange="reversed", color=SECOND),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, height=400,
    margin=dict(l=100, r=20, t=80, b=86), bargap=0.35,
    legend=dict(orientation="h", y=-0.22, x=0.0, xanchor="left", yanchor="top",
                font=dict(size=11, color=SECOND)),
)
fig.show()

Three different things sit in that chart, and they call for three different
responses.

**Settled.** The four edges into and out of `growth` are full bars of a single
colour: present in every resample and pointing the same way in nearly all of
them. Nothing more to do.

**Present, and undecidable.** `heat–light` is a full bar that is mostly aqua —
the data is certain the edge is there and cannot tell which way it runs. That
is not instability. It is the equivalence class showing through, and *no number
of extra rows will change it*, because both orientations imply exactly the same
independencies.

**Fringe.** `light–water`, `light–yield` and `heat–water` show up in one
resample in eight or fewer. Those are the search reacting to noise, and a single
run would have presented one of them as a finding.

In [ ]:
print("stable at 80%      :", [(e.a, e.b) for e in stability.stable(0.8)])
print("contested          :", [(e.a, e.b) for e in stability.contested()])
print("there but unoriented:", [(e.a, e.b) for e in stability.undecided_direction()])

## Which experiment settles them

An edge the data cannot orient is a question for an intervention, and
`orientation_gain` prices each candidate *before* anything is run. Randomizing
a variable severs its incoming edges, so an edge with exactly one endpoint in
the target has its direction revealed — and an edge with **both** endpoints
inside does not, which is why the biggest experiment is not the best one.

In [ ]:
from axiom.discover import orientation_gain
from axiom.identify import CausalGraph

hypothesis = CausalGraph.from_edges(
    "light -> heat, light -> growth, heat -> growth, water -> growth, growth -> yield"
)
candidates = [["light"], ["heat"], ["growth"], ["water"], ["light", "heat"]]
gains = [orientation_gain(hypothesis, [t]) for t in candidates]
table(
    [
        [str(target), len(gain), str([f"{a}→{b}" for a, b in gain]) if gain else "— nothing new"]
        for target, gain in zip(candidates, gains)
    ],
    headers=("randomize", "orients", "which"),
)

In [ ]:
counts = {", ".join(t): len(g) for t, g in zip(candidates, gains)}
fig = compare(
    [f"randomize {k}" for k in counts], list(counts.values()),
    highlight=f"randomize {max(counts, key=counts.get)}",
    value_fmt="{:.0f}",
    title="What each experiment would buy, in edges",
    subtitle="edges the intervention would orient that observation could not",
    x_title="edges newly oriented",
)
caption(fig, "Randomizing both ends of an edge reveals nothing about it, which is why the "
             "largest experiment is not the best one.")

Randomizing `light` settles the one edge observation could not — and so does
randomizing `heat`, since either end of that edge will do.

Randomizing `growth` or `water` settles **nothing**: every edge touching them
was already oriented by a collider, so that experiment buys no structure at all.

And randomizing `light` *and* `heat` together also settles nothing. Both ends of
the only undecided edge are inside the target, so the experiment destroys the
very comparison that would have oriented it. The largest experiment is the
worst one here, and the ranking says so before any budget is spent.

## The assumption underneath all of it

`ges` and `gies` assume **causal sufficiency**: that nothing unmeasured drives
two measured variables. Everything above rests on it, and it is exactly the
assumption that is usually false.

`fci` drops it. Its output is a **PAG**, whose edges carry a mark at each end
from three possibilities — and the third one is the point:

| mark | reads | means |
|---|---|---|
| arrowhead | `x *→ y` | `y` is **not** a cause of `x` |
| tail | `x —* y` | `x` **is** a cause of `y` |
| circle | `x o—* y` | the data does not determine which |

So `x <-> y` says *neither causes the other* — something you did not measure
drives both. A CPDAG has no way to say that.

In [ ]:
from axiom.discover import PAG, Mark, PagEdge, fci, oracle_independence

# A trial where soil quality was never recorded. It drives both sprouting and
# yield; seed lot and drainage were recorded and each affects only one of them.
world = CausalGraph.from_edges(
    "seed -> sprout, drain -> harvest, soil -> sprout, soil -> harvest",
    unmeasured=["soil"],
)
observed = ["seed", "drain", "sprout", "harvest"]
pag: PAG = fci(observed, oracle_independence(world))
print(pag.to_text())
print(pag.summary())
rows = []
for edge in pag.edges:
    assert isinstance(edge, PagEdge)
    left: Mark = edge.mark_a
    rows.append([edge.to_text(), f"{edge.a}: {left}", f"{edge.b}: {edge.mark_b}"])
table(rows, headers=("edge", "mark at one end", "mark at the other"))

In [ ]:
PAG_POSITIONS = {
    "seed": (0.0, 1.1), "sprout": (1.0, 1.1),
    "drain": (0.0, 0.0), "harvest": (1.0, 0.0),
}
pag_figure(
    PAG_POSITIONS, pag,
    "What the data can say when soil quality was never recorded",
    subtitle="Red with two arrowheads = confounded. Grey circles = the data did not decide.",
    height=380,
).show()

`sprout <-> harvest` in red is the finding: **neither causes the other**, and
something unmeasured drives both. That is a conclusion, not a shrug — and it is
the conclusion a CPDAG would have had to render as a plain edge, inviting
someone to read it as causal.

The circles are the honest remainder. FCI here places every mark it can and
leaves the rest open; `limits_hit` says which of Zhang's rules are not
implemented, and an unimplemented rule leaves circles rather than wrong marks.

In [ ]:
print("what this run did not attempt:")
table([[limit] for limit in pag.limits_hit], headers=("limit the search hit",))

## The same thing, from data rather than an oracle

`oracle_independence` answers from a known graph, which is how the algorithm
gets tested apart from the statistics. `fci_from_data` uses a real test — and
finds the same confounding, given enough rows.

In [ ]:
from axiom.discover import IndependenceResult, PartialCorrelation, fci_from_data

rows = 6000
rng2 = np.random.default_rng(4)
soil = rng2.normal(size=rows)
seed = rng2.normal(size=rows)
drain = rng2.normal(size=rows)
sprout = 1.2 * seed + 1.4 * soil + rng2.normal(size=rows)
harvest = 1.1 * drain + 1.3 * soil + rng2.normal(size=rows)
measured = Dataset.observational(
    np.column_stack([seed, drain, sprout, harvest]), ["seed", "drain", "sprout", "harvest"]
)

# The test underneath, on its own: seed and harvest are unrelated.
check = PartialCorrelation(measured).test("seed", "harvest")
assert isinstance(check, IndependenceResult)
print(check.describe())

from_data = fci_from_data(measured, alpha=0.01)
print("\nPAG from data:", from_data.to_text())
print("confounded pairs:", [e.to_text() for e in from_data.bidirected])

## A discovered graph is a hypothesis

Which puts us back at the start of the loop:

    draw  ->  refute the structure  ->  repair  ->  identify  ->
    estimate  ->  refute the estimate  ->  decide

A graph out of `ges` or `fci` enters that loop at the same place a hand-drawn
one does — as a claim to be tested, not a result. Hand it to
`diagnose.refute_structure` and it will make the same checkable claims;
hand it to `identify_effect` and it will tell you what, if anything, the graph
licenses. Nothing here promotes a discovered graph above a drawn one. It just
means you no longer have to start from a blank page, or take one on trust.

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
import numpy as np

from axiom.discover import Dataset, edge_stability
from axiom.display import show
from axiom.viz import stability

rng = np.random.default_rng(0)
n = 400
a = rng.normal(size=n)
b = 1.4 * a + rng.normal(size=n)
d = 0.8 * a + rng.normal(size=n)
c = -0.9 * b + 1.1 * d + rng.normal(size=n)

data = Dataset.observational(np.column_stack([a, b, c, d]), ["a", "b", "c", "d"])
report = edge_stability(data, n_bootstrap=25, seed=1)
show(report)
stability(report)

Two of these four edges are oriented and two are not, and the collider is the difference. `b → c ← d` is a v-structure, so the resamples settle both of its directions; `a – b` and `a – d` are adjacent in *every* resample and oriented in none. That is not weak evidence — it is the strongest evidence observation can give, and it does not include a direction. More rows will not move those two bars; only an intervention will. The two short bars are the different failure: edges the data is unsure are there at all.